<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/BART_ZeroShot_DRS_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs best on a GPU runtime (e.g., T4 GPU).

# Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import math
import os
import sys
from transformers import pipeline

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

# Read in data

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

/content/digi-inno-road-prod/innoprod/wrangling/wrangling_tools.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series.loc[mask] = new_value


# Transform data

In [ ]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
qual_df = roadmaps_df[['Client ID', 'Current Digital Readiness Score (refer to PAS:1040)'] + qual_cols].copy()
qual_df['Context'] = roadmaps_df.apply(lambda row: ' '.join([row[c] for c in qual_cols]), axis=1)
qual_df.drop(qual_cols, axis=1, inplace=True)
# qual_df


# Run pipeline

In [ ]:
model_name = "facebook/bart-large-mnli"
classifier = pipeline("zero-shot-classification", model=model_name)
hypothesis_template = ("This company's digital readiness level is best described as: {}")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
context = qual_df['Context'].to_list()

results = classifier(
      context,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)

In [ ]:
qual_df['Predicted DRS'] = [DRS_LEVELS.index(rd['labels'][0])+1 for rd in results]
qual_df['Probability'] = [rd['scores'][0] for rd in results]
qual_df['Confidence'] = [abs(math.log(rd['scores'][0]) - math.log(rd['scores'][1])) for rd in results]

In [ ]:
qual_df = qual_df.drop(columns='Context')

In [ ]:
qual_df

,Client ID,Current Digital Readiness Score (refer to PAS:1040),Predicted DRS,Probability,Confidence
0,65CE53AE-8233-D485-957C-705AF17CCE77,4,4,0.222960,0.259758
1,00CCFCC3-97CA-83F5-9E32-E34EBB545EE9,6,4,0.189638,0.290997
2,eb08ecff-5bf9-e60a-9cbf-66e8050adf5a,3,5,0.293619,0.049541
3,0C163899-5F59-970B-1033-ADCCE41DB525,5,8,0.187420,0.308783
4,C713DA4C-F07B-F227-351A-5C376EC69358,5,2,0.379620,1.155108
...,...,...,...,...,...
215,b6060463-e942-8a4a-77be-67caff694cdf,6,5,0.366575,0.709447
216,8bf25d49-f732-82c8-1274-67dd87ab56e9,3,4,0.163879,0.030665
217,63ebf90a-ed7d-e34f-4033-67b30f6d19f4,<NA>,7,0.573949,1.645783
218,8dbafede-6e05-b0ed-af96-67dd8480f482,4,4,0.178289,0.022713


# Write results

In [ ]:
if IN_COLAB:
  output_dir = os.path.join(dirpath, 'analysis', 'outputs')

qual_df.to_csv(os.path.join(output_dir, 'BART_DRS_results.csv'), index=False)